# Visualization Notebook

Loads the `latent_data.npz` saved by `experiment_runner.ipynb` and renders
interactive 3D PCA and UMAP plots (Plotly).  Use this when you want to
re-visualize results without re-training.

In [ ]:
# ============================================================
# CONFIGURATION
# ============================================================

EXPERIMENT_ID = 1   # Must match the experiment_runner output
DATA_DIR = f"outputs/experiment_{EXPERIMENT_ID}"

In [ ]:
!pip install -q plotly

In [ ]:
import os, io, base64
import numpy as np
from PIL import Image
import plotly.graph_objects as go
from IPython.display import HTML, display

# Load saved data
data = np.load(os.path.join(DATA_DIR, 'latent_data.npz'), allow_pickle=True)

all_latents = data['latents']
all_images  = data['images']
pca_3d      = data['pca_3d']
umap_3d     = data['umap_3d']
pca_var     = data['pca_var']
exp_id      = int(data['experiment'][0])
img_size    = int(data['image_size'][0])

print(f"Loaded experiment {exp_id} | {len(all_latents)} samples | "
      f"latent dim={all_latents.shape[1]} | image {img_size}x{img_size}")

In [ ]:
def build_3d_figure(coords_3d, all_imgs, method_name, axis_labels,
                    title_extra='', size=64):
    """
    Build an interactive Plotly 3D scatter with hover-image display.
    Colors are mapped to the x-axis values using Viridis.
    Returns (plotly Figure, full HTML string).
    """

    def _to_b64(arr):
        img = Image.fromarray((arr.reshape(size, size) * 255).astype(np.uint8))
        img = img.resize((200, 200), Image.LANCZOS)
        buf = io.BytesIO()
        img.save(buf, format='PNG')
        return 'data:image/png;base64,' + base64.b64encode(buf.getvalue()).decode()

    b64 = [_to_b64(im) for im in all_imgs]
    x, y, z = coords_3d[:, 0], coords_3d[:, 1], coords_3d[:, 2]

    fig = go.Figure(go.Scatter3d(
        x=x, y=y, z=z,
        mode='markers',
        marker=dict(
            size=4, opacity=0.7,
            color=x,
            colorscale='Viridis',
            colorbar=dict(title=axis_labels[0]),
            line=dict(width=0.3, color='white')
        ),
        customdata=np.column_stack([x, y, z, b64]),
        hovertemplate=(
            '<b>Sample #%{pointNumber}</b><br>'
            f'<b>{axis_labels[0]}:</b>' + ' %{customdata[0]}<br>'
            f'<b>{axis_labels[1]}:</b>' + ' %{customdata[1]}<br>'
            f'<b>{axis_labels[2]}:</b>' + ' %{customdata[2]}<br>'
            '<extra></extra>'
        ),
        showlegend=False
    ))

    fig.update_layout(
        title=dict(
            text=f'3D {method_name} of Latent Space '
                 f'({all_latents.shape[1]}D -> 3D){title_extra}',
            x=0.5, xanchor='center', font=dict(size=18)
        ),
        scene=dict(
            xaxis_title=axis_labels[0],
            yaxis_title=axis_labels[1],
            zaxis_title=axis_labels[2],
            camera=dict(eye=dict(x=1.5, y=1.5, z=1.5)),
            bgcolor='#f8f9fa'
        ),
        width=1000, height=800,
        template='plotly_white'
    )

    div_id = f'plot3d_{method_name.lower().replace(" ","_")}'
    func_id = div_id.replace('-', '_')
    plot_html = fig.to_html(include_plotlyjs='cdn', div_id=div_id)
    full_html = f"""
<div id="container_{div_id}" style="display:flex;align-items:flex-start;gap:24px;">
    <div style="flex:1;min-width:0;">{plot_html}</div>
    <div id="imgdiv_{div_id}" style="width:280px;min-height:300px;display:flex;align-items:center;justify-content:center;flex-shrink:0;">
        <p style="color:#666;text-align:center;">Hover over a point to see its image.</p>
    </div>
</div>
<script>
var p = document.getElementById('{div_id}');
function showImg_{func_id}(data) {{
    var pt = data.points[0];
    var cd = pt.customdata;
    document.getElementById('imgdiv_{div_id}').innerHTML =
        '<div style="padding:20px;background:#f0f0f0;border-radius:12px;text-align:center;">' +
        '<h3 style="margin:0 0 10px 0;color:#333;">Sample #' + pt.pointNumber + '</h3>' +
        '<img src="' + cd[3] + '" width="200" height="200" style="border:2px solid #444;border-radius:6px;display:block;margin:0 auto;">' +
        '<div style="margin-top:10px;font-size:12px;color:#555;line-height:1.6;">' +
        '{axis_labels[0]}: ' + Number(cd[0]).toFixed(4) + '<br>' +
        '{axis_labels[1]}: ' + Number(cd[1]).toFixed(4) + '<br>' +
        '{axis_labels[2]}: ' + Number(cd[2]).toFixed(4) + '</div></div>';
}}
p.on('plotly_hover',  showImg_{func_id});
p.on('plotly_click',  showImg_{func_id});
</script>
"""
    return fig, full_html

print("build_3d_figure() defined.")

---
## PCA 3D

In [ ]:
pca_labels = [
    f'PC1 ({pca_var[0]:.1%})',
    f'PC2 ({pca_var[1]:.1%})',
    f'PC3 ({pca_var[2]:.1%})'
]

pca_fig, pca_html = build_3d_figure(
    pca_3d, all_images, 'PCA', pca_labels,
    title_extra=f' | Var={pca_var.sum():.1%}', size=img_size
)

display(HTML(pca_html))

---
## UMAP 3D

In [ ]:
umap_labels = ['UMAP1', 'UMAP2', 'UMAP3']

umap_fig, umap_html = build_3d_figure(
    umap_3d, all_images, 'UMAP', umap_labels, size=img_size
)

display(HTML(umap_html))

---

You can also open the saved HTML files directly in a browser:
- `outputs/experiment_X/pca_3d.html`
- `outputs/experiment_X/umap_3d.html`